In [1]:
import sys
import pandas as pd
import praw

print("Python executable:", sys.executable)
print("Pandas version:", pd.__version__)
print("PRAW version:", praw.__version__)

Python executable: c:\Users\rakas\OneDrive\Documents\Github\Reddit-product-intelligence\Reddit-product-intelligence\.venv\Scripts\python.exe
Pandas version: 3.0.6
PRAW version: 8.0.3


In [2]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
SAMPLE_PATH = PROJECT_ROOT / "data" / "sample" / "sample_comments.csv"

comments_df = pd.read_csv(SAMPLE_PATH)
comments_df

,comment_id,post_id,body,score,created_utc,subreddit
0,c001,p001,Kirkland protein bars are my favorite Costco s...,42,2026-09-01T12:00:00Z,Costco
1,c002,p001,Bitchin Sauce is excellent with chips and vege...,31,2026-09-01T13:00:00Z,Costco
2,c003,p002,"I love the Bare chicken nuggets, especially fo...",27,2026-09-02T11:30:00Z,Costco
3,c004,p002,The Kirkland Colombian coffee beans are good v...,19,2026-09-02T12:15:00Z,Costco
4,c005,p003,Avoid the Kirkland protein bars if you dislike...,8,2026-09-03T09:00:00Z,Costco
5,c006,p003,Bitchin Sauce is overpriced at my warehouse.,5,2026-09-03T10:20:00Z,Costco
6,c007,p004,Fairlife protein shakes are worth buying when ...,36,2026-09-04T14:00:00Z,Costco
7,c008,p004,Kirkland paper towels are reliable and a good ...,24,2026-09-04T15:30:00Z,Costco
8,c009,p005,I did not like the Bare nuggets; they were too...,6,2026-09-05T16:00:00Z,Costco
9,c010,p005,Kirkland pesto is delicious and I buy it regul...,22,2026-09-05T17:45:00Z,Costco


In [3]:
comments_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   comment_id   10 non-null     str  
 1   post_id      10 non-null     str  
 2   body         10 non-null     str  
 3   score        10 non-null     int64
 4   created_utc  10 non-null     str  
 5   subreddit    10 non-null     str  
dtypes: int64(1), str(5)
memory usage: 1.5 KB


In [4]:
comments_df["score"].describe()


count    10.000000
mean     22.000000
std      12.719189
min       5.000000
25%      10.750000
50%      23.000000
75%      30.000000
max      42.000000
Name: score, dtype: float64

In [5]:
comments_df.sort_values("score", ascending=False)[
    ["comment_id", "score", "body"]
]

,comment_id,score,body
0,c001,42,Kirkland protein bars are my favorite Costco s...
6,c007,36,Fairlife protein shakes are worth buying when ...
1,c002,31,Bitchin Sauce is excellent with chips and vege...
2,c003,27,"I love the Bare chicken nuggets, especially fo..."
7,c008,24,Kirkland paper towels are reliable and a good ...
9,c010,22,Kirkland pesto is delicious and I buy it regul...
3,c004,19,The Kirkland Colombian coffee beans are good v...
4,c005,8,Avoid the Kirkland protein bars if you dislike...
8,c009,6,I did not like the Bare nuggets; they were too...
5,c006,5,Bitchin Sauce is overpriced at my warehouse.


In [ ]:
'''Comment text analysis: tokenization, n-grams, removal of stop words,
 ngram phrases, and phrase frequency counting.'''

import re
from collections import Counter

STOP_WORDS = {
    "a", "an", "and", "are", "as", "at", "be", "because", "but",
    "by", "for", "from", "has", "have", "i", "if", "in", "is",
    "it", "its", "my", "of", "on", "or", "so", "that", "the",
    "they", "this", "to", "too", "was", "were", "with", "you",
    "your",
}

def tokenize(text: str) -> list[str]:
    text = text.lower()
    return re.findall(r"[a-z0-9']+", text)

def make_ngrams(tokens: list[str], n: int) -> list[str]:
    return [
        " ".join(tokens[i:i + n])
        for i in range(len(tokens) - n + 1)
    ]

all_phrases = []

for comment in comments_df["body"].dropna():
    tokens = [
        token
        for token in tokenize(comment)
        if token not in STOP_WORDS and len(token) > 1
    ]

    all_phrases.extend(make_ngrams(tokens, 2))
    all_phrases.extend(make_ngrams(tokens, 3))

phrases_df = (
    pd.DataFrame(
        Counter(all_phrases).items(),
        columns=["phrase", "mentions"],
    )
    .query("mentions >= 1")
    .sort_values(["mentions", "phrase"], ascending=[False, True])
    .reset_index(drop=True)
)

phrases_df.head(30)

,phrase,mentions
0,bitchin sauce,2
1,kirkland protein,2
2,kirkland protein bars,2
3,protein bars,2
4,avoid kirkland,1
5,avoid kirkland protein,1
6,bare chicken,1
7,bare chicken nuggets,1
8,bare nuggets,1
9,bare nuggets salty,1


In [7]:
products_df = pd.read_csv(PROJECT_ROOT / "config" / "validated_products.csv")
products_df

,product_id,brand,product_name,category,aliases
0,kirkland_protein_bars,Kirkland Signature,Kirkland Protein Bars,snacks,kirkland protein bars|kirkland bars
1,bitchin_sauce,Bitchin' Sauce,Bitchin' Sauce,dips,bitchin sauce|bitchin' sauce
2,bare_chicken_nuggets,Bare,Bare Chicken Nuggets,frozen_food,bare chicken nuggets|bare nuggets
3,kirkland_colombian_coffee,Kirkland Signature,Kirkland Colombian Coffee,coffee,kirkland colombian coffee|kirkland colombian|c...
4,fairlife_protein_shakes,Fairlife,Fairlife Protein Shakes,beverages,fairlife protein shakes|fairlife shakes
5,kirkland_paper_towels,Kirkland Signature,Kirkland Paper Towels,household,kirkland paper towels
6,kirkland_pesto,Kirkland Signature,Kirkland Pesto,food,kirkland pesto


In [8]:
def find_product_mentions(comment_text: str, products: pd.DataFrame) -> list[dict]:
    text = comment_text.lower()
    matches = []

    for _, product in products.iterrows():
        aliases = product["aliases"].lower().split("|")

        for alias in aliases:
            if alias in text:
                matches.append(
                    {
                        "product_id": product["product_id"],
                        "brand": product["brand"],
                        "product_name": product["product_name"],
                        "category": product["category"],
                        "matched_alias": alias,
                    }
                )
                break

    return matches

In [9]:
mention_records = []

for _, comment in comments_df.iterrows():
    matches = find_product_mentions(comment["body"], products_df)

    for match in matches:
        mention_records.append(
            {
                "comment_id": comment["comment_id"],
                "post_id": comment["post_id"],
                "body": comment["body"],
                "comment_score": comment["score"],
                "created_utc": comment["created_utc"],
                **match,
            }
        )

mentions_df = pd.DataFrame(mention_records)
mentions_df

,comment_id,post_id,body,comment_score,created_utc,product_id,brand,product_name,category,matched_alias
0,c001,p001,Kirkland protein bars are my favorite Costco s...,42,2026-09-01T12:00:00Z,kirkland_protein_bars,Kirkland Signature,Kirkland Protein Bars,snacks,kirkland protein bars
1,c002,p001,Bitchin Sauce is excellent with chips and vege...,31,2026-09-01T13:00:00Z,bitchin_sauce,Bitchin' Sauce,Bitchin' Sauce,dips,bitchin sauce
2,c003,p002,"I love the Bare chicken nuggets, especially fo...",27,2026-09-02T11:30:00Z,bare_chicken_nuggets,Bare,Bare Chicken Nuggets,frozen_food,bare chicken nuggets
3,c004,p002,The Kirkland Colombian coffee beans are good v...,19,2026-09-02T12:15:00Z,kirkland_colombian_coffee,Kirkland Signature,Kirkland Colombian Coffee,coffee,kirkland colombian coffee
4,c005,p003,Avoid the Kirkland protein bars if you dislike...,8,2026-09-03T09:00:00Z,kirkland_protein_bars,Kirkland Signature,Kirkland Protein Bars,snacks,kirkland protein bars
5,c006,p003,Bitchin Sauce is overpriced at my warehouse.,5,2026-09-03T10:20:00Z,bitchin_sauce,Bitchin' Sauce,Bitchin' Sauce,dips,bitchin sauce
6,c007,p004,Fairlife protein shakes are worth buying when ...,36,2026-09-04T14:00:00Z,fairlife_protein_shakes,Fairlife,Fairlife Protein Shakes,beverages,fairlife protein shakes
7,c008,p004,Kirkland paper towels are reliable and a good ...,24,2026-09-04T15:30:00Z,kirkland_paper_towels,Kirkland Signature,Kirkland Paper Towels,household,kirkland paper towels
8,c009,p005,I did not like the Bare nuggets; they were too...,6,2026-09-05T16:00:00Z,bare_chicken_nuggets,Bare,Bare Chicken Nuggets,frozen_food,bare nuggets
9,c010,p005,Kirkland pesto is delicious and I buy it regul...,22,2026-09-05T17:45:00Z,kirkland_pesto,Kirkland Signature,Kirkland Pesto,food,kirkland pesto


In [ ]:
POSITIVE_WORDS = [
    "recommend",
    "favorite",
    "great",
    "excellent",
    "love",
    "good value",
    "worth buying",
    "worth it",
    "reliable",
    "delicious",
    "yummy",
    "yum",
    "tasty",
    "bang for the buck"
]

NEGATIVE_WORDS = [
    "avoid",
    "bad",
    "hate",
    "terrible",
    "disappointing",
    "overpriced",
    "not like",
    "did not like",
    "too salty",
    "awful",
    "disgusting",
    "waste of money",
    "returned",
    
]